# Fotos + Textbeschreibung → STL

Allgemeine Pipeline der Gruppe 7 (Objektrekonstruktion) für **eigene Fotos eines beliebigen Objekts**:

```
Fotos (30–45, Rundgang um das Objekt) + Text ("a figurine.")
  → COLMAP (Kameraposen)
  → Grounding DINO + SAM2 (Maske des genannten Objekts pro Foto)
  → 3D Gaussian Splatting (nerfstudio splatfacto, nur maskierte Pixel)
  → Masken-Carving der Gaussians (Visual Hull) → Poisson-Mesh → STL
```

Für ein neues Objekt nur die zwei Parameter in Abschnitt 3 ändern: `DRIVE_PHOTOS` und `OBJECT_PROMPT`.

**Erster Lauf (2026-10-25):** 3D-gedruckte Figur, 43 Smartphone-Fotos (iPhone, 4032×3024) — COLMAP 43/43 registriert, Masken 43/43.
Alle Abschnitte auf Colab T4 ausgeführt → `figurine_object_carved.stl`.

**Aufnahmeregeln** (bewährt): Objekt bleibt stehen, man geht selbst mit der Kamera herum (nicht das Objekt drehen);
2–3 Runden auf verschiedenen Höhen (seitlich, ~45° von oben, fast von oben); texturierter Untergrund (kein weißes Papier);
gleichmäßiges Licht, kein Blitz, kein Zoom; Fotos als **Datei** (Original-JPEG) übertragen, nicht über Messenger komprimiert.

Läuft auf Google Colab mit GPU (T4). nerfstudio läuft in einem eigenen venv (`/content/nsenv`, Python 3.10),
weil das Colab-Kernel (Python 3.13 / torch 2.x) mit nerfstudio 1.1.5 nicht kompatibel ist.

## 1. GPU prüfen

In [ ]:
!nvidia-smi -L

## 2. Umgebung (venv + COLMAP)

~30 min beim ersten Mal; überspringt sich selbst, wenn das venv schon existiert.
Versionen sind exakt so gewählt, dass `gsplat` als vorkompiliertes Wheel installiert wird (Quell-Build läuft auf Colab aus dem RAM).
`numpy==1.26.4` muss zuletzt installiert werden (sonst zieht nerfstudio numpy 2.x, und torch 2.1 bricht).

In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python
if [ ! -x /content/nsenv/bin/ns-train ]; then
  pip install -q uv
  uv python install 3.10
  uv venv --python 3.10 /content/nsenv
  uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
  uv pip install --python $NSPY "setuptools<70" wheel
  uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
  uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
  uv pip install --python $NSPY "numpy==1.26.4"
fi
apt-get -qq install -y colmap > /dev/null
$NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
which colmap

## 3. Parameter + Fotos laden

Fotos liegen als Ordner auf Google Drive. Sie werden nach EXIF gedreht (iPhone speichert Hochformat als gedrehtes Querformat + EXIF-Flag)
und auf max. 1600 px verkleinert — COLMAP läuft hier auf der CPU, 12-MP-Bilder wären sehr langsam.
Defekte Vorschaubilder (< 1000 px) werden übersprungen.

In [ ]:
DRIVE_PHOTOS  = '/content/drive/MyDrive/Applied_Robotics/pictures'   # Ordner mit den Fotos
OBJECT_PROMPT = 'a figurine.'                                        # Textbeschreibung des Objekts
WORK = '/content/figurine'                                           # Arbeitsordner auf Colab

from google.colab import drive
drive.mount('/content/drive')

import glob, os
from PIL import Image, ImageOps
RAW = f'{WORK}/raw'
os.makedirs(RAW, exist_ok=True)
n = 0
for f in sorted(glob.glob(DRIVE_PHOTOS + '/*.JPG') + glob.glob(DRIVE_PHOTOS + '/*.jpg')):
    im = Image.open(f)
    if max(im.size) < 1000:
        continue
    im = ImageOps.exif_transpose(im).convert('RGB')
    im.thumbnail((1600, 1600), Image.LANCZOS)
    im.save(f'{RAW}/{os.path.splitext(os.path.basename(f))[0]}.jpg', quality=95)
    n += 1
print('Fotos:', n, im.size)

## 4. Kameraposen (COLMAP)

~15 min (CPU, exhaustive matching). `QT_QPA_PLATFORM=offscreen` und `--no-gpu` sind nötig, weil das apt-COLMAP
in Colab weder Display noch CUDA/OpenGL hat. Figur: *COLMAP found poses for all images* (43/43).

In [ ]:
!rm -rf {WORK}/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data {WORK}/raw --output-dir {WORK}/processed \
  --matching-method exhaustive --no-gpu 2>&1 | tail -5

## 5. Sprachgesteuerte Segmentierung (Grounding DINO + SAM2)

Grounding DINO findet aus `OBJECT_PROMPT` eine Box, SAM2 macht daraus eine pixelgenaue Maske.
Masken (`masks/<frame>.png`, 255 = Objekt) gehen ins Training — der Hintergrund wird nie trainiert.
Unten wird eine Vorschau angezeigt (Objekt hell, Hintergrund abgedunkelt). Figur: 43/43 Masken, 0 ohne Box.

In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = 'cuda'
GD, SAM = 'IDEA-Research/grounding-dino-base', 'facebook/sam2-hiera-large'
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()

def gdino_boxes(img, text, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors='pt').to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r['boxes'].cpu().numpy(), r['scores'].cpu().numpy()

_pp = getattr(sam_proc, 'post_process_masks', None) or sam_proc.image_processor.post_process_masks
def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors='pt').to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    m = _pp(out.pred_masks.cpu(), inp['original_sizes'].cpu())[0].numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    return np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])]).any(0)

D = f'{WORK}/processed'
IMG_DIR, MASK_DIR = f'{D}/images', f'{D}/masks'
os.makedirs(MASK_DIR, exist_ok=True)
names = sorted(os.listdir(IMG_DIR))
no_box, previews = [], []
for i, n in enumerate(tqdm(names)):
    img = Image.open(f'{IMG_DIR}/{n}').convert('RGB')
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, 'toy. statue. object.', 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box.append(n); mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f'{MASK_DIR}/{os.path.splitext(n)[0]}.png')
    if i % 5 == 0:
        ov = np.array(img); ov[~mask] = (ov[~mask]*0.2).astype(np.uint8)
        previews.append(Image.fromarray(ov).resize((240, 320)))
print('Masken:', len(os.listdir(MASK_DIR)), '| ohne Box:', len(no_box), no_box)

sheet = Image.new('RGB', (240*len(previews), 320))
for k, p in enumerate(previews): sheet.paste(p, (240*k, 0))
display(sheet)

## 6. Training: 3D Gaussian Splatting mit Masken

splatfacto, 7000 Iterationen (~10 min auf T4). COLMAP-Dataparser, damit `--masks-path` sicher greift.

In [ ]:
%cd {WORK}
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto \
  --data {WORK}/processed \
  --max-num-iterations 7000 --steps-per-save 2000 \
  --vis tensorboard --viewer.quit-on-train-completion True \
  --output-dir outputs_fig --pipeline.model.cull-alpha-thresh 0.1 \
  colmap --images-path images --colmap-path colmap/sparse/0 --masks-path masks --downscale-factor 1 2>&1 | tail -20

## 7. Gaussians → Masken-Carving → Mesh → STL

**Problem ohne Carving:** splatfacto initialisiert die Gaussians aus der COLMAP-Punktwolke — die enthält den ganzen Raum
(Boden, Möbel). Die Masken verhindern nur, dass diese Hintergrund-Gaussians *trainiert* werden; sie bleiben aber
in der Szene. Beim truck trennte DBSCAN sie ab, bei der Figur waren Boden und Figur zu einem Cluster verwachsen
(Mesh = Figur in einer Wolke aus Schwebeteilchen, Bounding Box 3,1 × 3,0 × 2,1).

**Lösung (Visual Hull):** jedes Gaussian-Zentrum wird in alle Trainingskameras projiziert; behalten wird es nur,
wenn es in ≥ 90 % der Ansichten, in denen es im Bild liegt, *innerhalb* der SAM2-Maske landet (und in ≥ 10 Ansichten
sichtbar ist, Opazität > 0,1). Kameras und Gaussians kommen aus derselben geladenen nerfstudio-Pipeline
(`eval_setup`), damit beide garantiert im selben Koordinatensystem sind.
Danach: Ausreißerfilter → Normalen (k-NN, skalenunabhängig) → Poisson (depth 9) → Aufräumen → STL.

Figur: 125 573 Gaussians → **31 038 nach Carving** → STL mit 102 842 Ecken / 199 253 Dreiecken,
Bounding Box 0,83 × 0,63 × 0,84 (Szeneneinheiten). Unterseite bleibt offen (nie fotografiert), `watertight=False`.

In [ ]:
import glob, os, subprocess
cfg = sorted(glob.glob(f'{WORK}/outputs_fig/*/splatfacto/*/config.yml'))[-1]
EXPORT = f'{WORK}/export'
os.makedirs(EXPORT, exist_ok=True)
subprocess.run(f'/content/nsenv/bin/ns-export gaussian-splat --load-config {cfg} --output-dir {EXPORT}', shell=True)

script = f'''
import numpy as np, torch, open3d as o3d, trimesh
from pathlib import Path
from PIL import Image
from nerfstudio.utils.eval_utils import eval_setup

_, pipe, _, _ = eval_setup(Path("{cfg}"), test_mode="test")
model, ds = pipe.model, pipe.datamanager.train_dataset
X = model.means.detach().cpu().numpy()
op = torch.sigmoid(model.opacities).detach().cpu().numpy().ravel()
print("Gaussians:", len(X))

cams = ds.cameras
c2w = cams.camera_to_worlds.cpu().numpy()
fx, fy = cams.fx.cpu().numpy().ravel(), cams.fy.cpu().numpy().ravel()
cx, cy = cams.cx.cpu().numpy().ravel(), cams.cy.cpu().numpy().ravel()
masks = ds._dataparser_outputs.mask_filenames
inside = np.zeros(len(X)); seen = np.zeros(len(X))
for k in range(len(c2w)):
    M = np.array(Image.open(masks[k])) > 127
    H, W = M.shape[:2]
    R, t = c2w[k][:, :3], c2w[k][:, 3]
    Pc = (X - t) @ R                      # OpenGL-Kamera: z zeigt nach hinten
    z = -Pc[:, 2]
    u = fx[k] * Pc[:, 0] / z + cx[k]
    v = -fy[k] * Pc[:, 1] / z + cy[k]
    ok = (z > 0) & (u >= 0) & (u < W) & (v >= 0) & (v < H)
    seen += ok
    inside[np.where(ok)[0]] += M[v[ok].astype(int), u[ok].astype(int)]
frac = np.where(seen > 0, inside / np.maximum(seen, 1), 0)
keep = (frac >= 0.9) & (seen >= 10) & (op > 0.1)
print("nach Masken-Carving:", keep.sum())

pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(X[keep]))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
o3d.io.write_point_cloud("{EXPORT}/carved.ply", pcd)
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "{EXPORT}/object_carved.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out); m.fill_holes(); m.fix_normals(); m.process(validate=True); m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("Bounding Box:", m.extents.round(3), "| watertight:", m.is_watertight)
'''
open('/content/carve.py', 'w').write(script)
r = subprocess.run(['/content/nsenv/bin/python', '/content/carve.py'], capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:] if r.returncode else '')

### Schnelle Kontrolle: drei Projektionen der Mesh-Ecken

In [ ]:
!pip install -q trimesh
import trimesh, numpy as np, matplotlib.pyplot as plt
m = trimesh.load(f'{EXPORT}/object_carved.stl')
p = m.vertices[np.random.choice(len(m.vertices), min(30000, len(m.vertices)), replace=False)]
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, (i, j), t in zip(ax, [(0, 1), (0, 2), (1, 2)], ['XY', 'XZ', 'YZ']):
    a.scatter(p[:, i], p[:, j], s=0.2, c=p[:, 3 - i - j], cmap='viridis'); a.set_title(t); a.set_aspect('equal')
plt.show()

## 8. Ergebnis auf Google Drive sichern

In [ ]:
import shutil
OUT_DRIVE = '/content/drive/MyDrive/Applied_Robotics/results'
os.makedirs(OUT_DRIVE, exist_ok=True)
shutil.copy(f'{EXPORT}/object_carved.stl', f'{OUT_DRIVE}/figurine_object_carved.stl')
shutil.copy(f'{EXPORT}/splat.ply', f'{OUT_DRIVE}/figurine_splat.ply')
print(os.listdir(OUT_DRIVE))